# Creating your own dataset

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
# Install required libraries and git-lfs (needed to push large files to the Hub).
# git-lfs = Git Large File Storage, used by Hugging Face for big dataset files.

!pip install datasets evaluate transformers[sentencepiece]
!apt install git-lfs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.


You will need to setup git, adapt your email and name in the following cell.

In [ ]:
# Log in to the Hugging Face Hub.
# This saves your API token locally so you can push datasets.
# You'll need a free account at huggingface.co to get a token.

!git config --global user.email "YourEmail"
!git config --global user.name "Your username"

You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.

In [3]:
# Make a test GET request to the GitHub Issues API.
# 'page=1&per_page=1' fetches just 1 issue from page 1.
# This is a dry run to check the API works before we fetch thousands of issues.

from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
# Check the HTTP status code of our request.
# 200 = success. Other codes: 404 = not found, 403 = rate limited, 401 = unauthorized.
# Always check this before using the response data.

!pip install requests

In [5]:
# Inspect the JSON payload of the response.
# The GitHub API returns a list of issue objects with many fields:
# 'title', 'body', 'number', 'state', 'pull_request', 'user', etc.

import requests

url = "https://api.github.com/repos/huggingface/datasets/issues?page=1&per_page=1"
response = requests.get(url)

In [6]:
# Set your GitHub Personal Access Token (PAT) for authenticated requests.
# Unauthenticated: 60 requests/hour limit.
# Authenticated: 5,000 requests/hour limit.
# ⚠️ NEVER share this token or commit it to git!

response.status_code

200

In [7]:
# Define a function to download all issues from a GitHub repository.
# It paginates through the API (100 issues per page) and handles rate limits.
# Results are saved to a .jsonl file (one JSON object per line).
# 'state=all' fetches both open AND closed issues.

response.json()

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/8084',
  'repository_url': 'https://api.github.com/repos/huggingface/datasets',
  'labels_url': 'https://api.github.com/repos/huggingface/datasets/issues/8084/labels{/name}',
  'comments_url': 'https://api.github.com/repos/huggingface/datasets/issues/8084/comments',
  'events_url': 'https://api.github.com/repos/huggingface/datasets/issues/8084/events',
  'html_url': 'https://github.com/huggingface/datasets/pull/8084',
  'id': 4113045931,
  'node_id': 'PR_kwDODunzps7MYMNy',
  'number': 8084,
  'title': 'Add reshard support for JSON Lines and CSV file formats',
  'user': {'login': 'muyihao',
   'id': 37872457,
   'node_id': 'MDQ6VXNlcjM3ODcyNDU3',
   'avatar_url': 'https://avatars.githubusercontent.com/u/37872457?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/muyihao',
   'html_url': 'https://github.com/muyihao',
   'followers_url': 'https://api.github.com/users/muyihao/followers',
   'following_url':

In [ ]:
# Call fetch_issues() to download all issues from the 🤗 Datasets repo.
# This may take several minutes depending on your internet speed.
# The result is saved to 'datasets-issues.jsonl'.

GITHUB_TOKEN = 'Github Token'  # Copy your GitHub token here
headers = {"Authorization": f"token {GITHUB_TOKEN}"}

In [15]:
# Load the downloaded JSONL file as a HuggingFace Dataset.
# 'split="train"' is needed because JSONL loading creates a single split.
# We now have 3019 rows — more than the ~1000 shown on GitHub because
# the API also returns pull requests (which GitHub treats as issues).

import time
import math
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm


def fetch_issues(
    owner="huggingface",
    repo="datasets",
    num_issues=10_000,
    rate_limit=5_000,
    issues_path=Path("."),
):
    if not issues_path.is_dir():
        issues_path.mkdir(exist_ok=True)

    batch = []
    all_issues = []
    per_page = 100  # Number of issues to return per page
    num_pages = math.ceil(num_issues / per_page)
    base_url = "https://api.github.com/repos"

    for page in tqdm(range(num_pages)):
        # Query with state=all to get both open and closed issues
        query = f"issues?page={page}&per_page={per_page}&state=all"
        issues = requests.get(f"{base_url}/{owner}/{repo}/{query}", headers=headers)
        batch.extend(issues.json())

        if len(batch) > rate_limit and len(all_issues) < num_issues:
            all_issues.extend(batch)
            batch = []  # Flush batch for next time period
            print(f"Reached GitHub rate limit. Sleeping for one hour ...")
            time.sleep(60 * 60 + 1)

    all_issues.extend(batch)
    df = pd.DataFrame.from_records(all_issues)
    df.to_json(f"{issues_path}/{repo}-issues.jsonl", orient="records", lines=True)
    print(
        f"Downloaded all the issues for {repo}! Dataset stored at {issues_path}/{repo}-issues.jsonl"
    )

In [16]:
# Randomly sample 3 issues to inspect the pull_request field.
# .shuffle(seed=666) randomizes order (seed for reproducibility).
# .select(range(3)) picks the first 3 after shuffling.
# We zip html_url and pull_request to compare them side-by-side.

# Depending on your internet connection, this can take several minutes to run...
fetch_issues()

  0%|          | 0/100 [00:00<?, ?it/s]

Reached GitHub rate limit. Sleeping for one hour ...


KeyboardInterrupt: 

In [17]:
# Add a new boolean column 'is_pull_request' to each row.
# If pull_request field is None → it's a real issue (False).
# If pull_request field has data → it's a PR (True).
# This lets us filter out PRs later.

issues_dataset = load_dataset("json", data_files="datasets-issues.jsonl", split="train")
issues_dataset

NameError: name 'load_dataset' is not defined

In [18]:
# Fetch comments for a specific issue by its number.
# The Comments endpoint returns a list of comment objects.
# We GET the URL and check the full response — each comment has 'body', 'user', timestamps, etc.

sample = issues_dataset.shuffle(seed=666).select(range(3))

# Print out the URL and pull request entries
for url, pr in zip(sample["html_url"], sample["pull_request"]):
    print(f">> URL: {url}")
    print(f">> Pull request: {pr}\n")

NameError: name 'issues_dataset' is not defined

In [ ]:
# Define get_comments() to extract just the comment text (body) for any issue.
# This calls the GitHub API for one issue and returns a list of comment strings.
# Tested on issue #2792 — should return 1 comment.

issues_dataset = issues_dataset.map(
    lambda x: {"is_pull_request": False if x["pull_request"] is None else True}
)

In [ ]:
# Add all comments to each issue using .map().
# For every issue, get_comments() fetches its comments from the API.
# ⚠️ This makes one API call per issue — may take a few minutes for thousands of issues.

issue_number = 2792
url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
response = requests.get(url, headers=headers)
response.json()

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/comments/897594128',
  'html_url': 'https://github.com/huggingface/datasets/pull/2792#issuecomment-897594128',
  'issue_url': 'https://api.github.com/repos/huggingface/datasets/issues/2792',
  'id': 897594128,
  'node_id': 'IC_kwDODunzps41gDMQ',
  'user': {'login': 'bhavitvyamalik',
   'id': 19718818,
   'node_id': 'MDQ6VXNlcjE5NzE4ODE4',
   'avatar_url': 'https://avatars.githubusercontent.com/u/19718818?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/bhavitvyamalik',
   'html_url': 'https://github.com/bhavitvyamalik',
   'followers_url': 'https://api.github.com/users/bhavitvyamalik/followers',
   'following_url': 'https://api.github.com/users/bhavitvyamalik/following{/other_user}',
   'gists_url': 'https://api.github.com/users/bhavitvyamalik/gists{/gist_id}',
   'starred_url': 'https://api.github.com/users/bhavitvyamalik/starred{/owner}{/repo}',
   'subscriptions_url': 'https://api.github.com/users/

In [ ]:
# Log in to the Hub again (needed to push the dataset).
# This step can be skipped if you already logged in above.

def get_comments(issue_number):
    url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
    response = requests.get(url, headers=headers)
    return [r["body"] for r in response.json()]


# Test our function works as expected
get_comments(2792)

["@albertvillanova my tests are failing here:\r\n```\r\ndataset_name = 'gooaq'\r\n\r\n    def test_load_dataset(self, dataset_name):\r\n        configs = self.dataset_tester.load_all_configs(dataset_name, is_local=True)[:1]\r\n>       self.dataset_tester.check_load_dataset(dataset_name, configs, is_local=True, use_local_dummy_data=True)\r\n\r\ntests/test_dataset_common.py:234: \r\n_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ \r\ntests/test_dataset_common.py:187: in check_load_dataset\r\n    self.parent.assertTrue(len(dataset[split]) > 0)\r\nE   AssertionError: False is not true\r\n```\r\nWhen I try loading dataset on local machine it works fine. Any suggestions on how can I avoid this error?"]

In [ ]:
# Push our augmented dataset to the Hugging Face Hub!
# push_to_hub() uploads it to your profile as 'your-username/github-issues'.
# Anyone in the community can now load your dataset.

# Depending on your internet connection, this can take a few minutes...
issues_with_comments_dataset = issues_dataset.map(
    lambda x: {"comments": get_comments(x["number"])}
)

In [ ]:
# Load the dataset back from the Hub to verify the upload worked.
# load_dataset() with a repo path ('lewtun/github-issues') downloads from the Hub.
# The dataset now has 2855 rows (fewer than 3019 because some entries were filtered).

from huggingface_hub import notebook_login

notebook_login()

In [ ]:
issues_with_comments_dataset.push_to_hub("github-issues")

In [ ]:
remote_dataset = load_dataset("lewtun/github-issues", split="train")
remote_dataset

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'pull_request', 'body', 'performed_via_github_app', 'is_pull_request'],
    num_rows: 2855
})